In [1]:
import torch

### set up CUDA as device if available
if torch.cuda.is_available():
    print("GPU is available")
    device = torch.device("cuda")
    cuda_id = torch.cuda.current_device()
    print(f"ID of current CUDA device:{torch.cuda.current_device()}")
    print(f"Name of current CUDA device:{torch.cuda.get_device_name(cuda_id)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("GPU is not available, using CPU")
    device = torch.device("cpu")
print(f"device: {device}")

GPU is available
cuda
CUDA version: 12.4
ID of current CUDA device:0
Name of current CUDA device:NVIDIA H200


In [2]:
import numpy as np
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm, trange
from IPython import display
from sklearn.metrics import average_precision_score
# from conf_and_plot import confusion_matrix_plots
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import KFold
from sklearn.model_selection import ParameterGrid
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import kruskal
import torch.nn.functional as F
from IPython.display import clear_output
import statistics

def seed_all(seed):
    if not seed:
        seed = 10

    print("[ Using Seed : ", seed, " ]")

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_all(2025)

[ Using Seed :  2025  ]


In [3]:
scint_thresh = 0.1 # set the phase scintillation threshold

### following variable not being used anywhere
scint_outlier_thresh = 5. # set the value that determines phase scintillation outliers (these data samples will be removed)

processed_data_2015 = pd.read_csv("processed_data_2015.csv")
processed_data_2015 = processed_data_2015.drop(processed_data_2015.columns[0], axis=1)
predicted_label = 'sigmaPhi projected to vertical at prediction time [radians]'
y = processed_data_2015[predicted_label].values
print(y.shape)

X_fSelect = processed_data_2015.drop(predicted_label, axis=1)
X_fSelect = X_fSelect.values
print(X_fSelect.shape)

(4465846,)
(4465846, 15)


In [4]:
class simple_timeSeries_CNN(nn.Module):
    def __init__(self, cnn_filters, fc_size, dropout_p, loss, sequence_length):
        super(simple_timeSeries_CNN, self).__init__()
        
        self.loss = loss
        self.sequence_length = sequence_length

        self.conv1 = nn.Conv1d(in_channels=15, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(num_features=cnn_filters)
        self.dropout = nn.Dropout(dropout_p)
        self.conv2 = nn.Conv1d(in_channels=cnn_filters, out_channels=cnn_filters*2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(num_features=cnn_filters*2)
        self.pool = nn.MaxPool1d(2)
        
        ### for simple, non true time-series CNN:
        # num_features = 25, MaxPool1d_size=2
        # 12 = round(25 / 2)
        # 6 = round(12 / 2)
        # self.fc1 = nn.Linear(cnn_filters * 2 * 6, fc_size)
        
        ### for true time-series CNN:
        # sequence_length, MaxPool1d_size=2
        # round(sequence_length / 2) = sequence_length // 2
        # per maxpool layer = sequence_length // 4
        
        self.fc1 = nn.Linear(cnn_filters * 2 * (sequence_length // 4), fc_size)
        # self.fc2 = nn.Linear(fc_size, 1)
        self.fc2 = nn.Linear(fc_size, sequence_length)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.dropout(x)
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1) # Flatten the output for the fully connected layers
        x = torch.relu(self.fc1(x))
        
        # second fully connected layer + reshape to [batch_size, sequence_length, 1]
        x = self.fc2(x).view(x.size(0), self.sequence_length, 1)

        if self.loss == 'focal':
            # x = torch.sigmoid(self.fc2(x))
            x = torch.sigmoid(x)
        elif self.loss == 'bce':
            # x = self.fc2(x)
            pass
        
        return x

In [5]:
class simple_timeSeries_CNN(nn.Module):
    def __init__(self, cnn_filters, fc_size, dropout_p, loss, sequence_length):
        super(simple_timeSeries_CNN, self).__init__()
        
        self.loss = loss
#         self.seq_length = sequence_length

        self.conv1 = nn.Conv1d(in_channels=15, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(num_features=cnn_filters)
        self.dropout = nn.Dropout(dropout_p)
        self.conv2 = nn.Conv1d(in_channels=cnn_filters, out_channels=cnn_filters*2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(num_features=cnn_filters*2)
        self.pool = nn.MaxPool1d(2)

        # Adjusted for sequence-to-one classification
        self.fc1 = nn.Linear(cnn_filters * 2 * (sequence_length // 4), fc_size)
        self.fc2 = nn.Linear(fc_size, 1)  # Output a single value per sequence

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.dropout(x)
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)  # Flatten the output for the fully connected layers
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)  # Output shape: (batch_size, 1)

        if self.loss == 'focal':
            x = torch.sigmoid(x)
        elif self.loss == 'bce':
            pass  # Assume BCEWithLogitsLoss applied externally

        return x

In [6]:
def true_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    tss = (TP / (TP + FN)) - (FP / (FP + TN))
    return tss

def heidke_skill_score(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    numerator = 2 * (TP * TN - FP * FN)
    denominator = (TP + FP) * (FP + TN) + (TP + FN) * (TN + FP)
    
    # Avoid division by zero
    if denominator == 0:
        return 0.0
    
    hss = numerator / denominator
    return hss

In [7]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction  # 'mean', 'sum', or 'none'

    def forward(self, inputs, targets):
        BCE_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        p_t = inputs * targets + (1 - inputs) * (1 - targets)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        elif self.reduction == 'none':
            return focal_loss
        else:
            raise ValueError(f"Invalid reduction method: {self.reduction}")

In [8]:
def train_model_with_cv(model, train_dataloader, val_dataloader, criterion, optimizer, scheduler, num_epochs):

    epochs = []
    training_loss = []
    validation_loss = []
    training_tss = []
    validation_tss = []
    training_hss = []
    validation_hss = []

    for epoch in range(num_epochs):

        ### training loop
        model.train()
        running_loss = 0.0
        predicted_training_labels = np.array([])
        y_train = np.array([])

        # for batch_inputs, batch_labels in tqdm(train_dataloader):
        for batch_inputs, batch_labels in train_dataloader:
            
            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            output = model(batch_inputs)
            
            output.to(device)
            
            loss = criterion(output, batch_labels.float())
            loss.backward()
            optimizer.step()
            
            if scheduler == False:
                pass
            else:
                scheduler.step()
            
            running_loss += loss.item()
            with torch.no_grad(): predicted_training_labels = np.append(predicted_training_labels, output.cpu())
            with torch.no_grad(): y_train = np.append(y_train, batch_labels.cpu())

        predicted_training_labels = np.where(predicted_training_labels > 0.1, 1, 0)

        train_loss = running_loss / len(train_dataloader)
        train_tss = true_skill_score(y_train.astype(int), predicted_training_labels.astype(int))
        train_hss = heidke_skill_score(y_train.astype(int), predicted_training_labels.astype(int))

        ### validation loop
        model.eval()
        with torch.no_grad():
            running_loss = 0.0
            predicted_val_labels = np.array([])
            y_val = np.array([])

            # for batch_inputs, labels in tqdm(val_dataloader):
            for batch_inputs, batch_labels in val_dataloader:
            
                batch_inputs = batch_inputs.to(device)
                batch_labels = batch_labels.to(device)
                
                outputs = model(batch_inputs)
            
                outputs.to(device)
                
                outputs = model(batch_inputs)
                loss = criterion(outputs, batch_labels.float())
                running_loss += loss.item()
                predicted_val_labels = np.append(predicted_val_labels, outputs.cpu())
                y_val = np.append(y_val, batch_labels.cpu())

        predicted_val_labels = np.where(predicted_val_labels > 0.1, 1, 0)

        val_loss = running_loss / len(val_dataloader)
        val_tss = true_skill_score(y_val.astype(int), predicted_val_labels.astype(int))
        val_hss = heidke_skill_score(y_val.astype(int), predicted_val_labels.astype(int))

        epochs.append(epoch)
        training_loss.append(train_loss)
        validation_loss.append(val_loss)
        training_tss.append(train_tss)
        validation_tss.append(val_tss)
        training_hss.append(train_hss)
        validation_hss.append(val_hss)

    return training_hss, validation_hss, training_tss, validation_tss


In [9]:
class CNNBinaryTimeSeriesClassifier():
    def __init__(self, cnn_filters=8, fc_size=64, sequence_length=60, dropout_p=0.1, batch_size=128, 
                 optimizer_type = 'adam', lr=0.001, sgd_momentum=0.9, weight_decay=0.0001,
                 loss='bce', bce_pos_class_weight=50, focal_alpha=0.25, focal_gamma=2,
                 num_epochs=5, cv_folds=5, scheduler_t=0):
        self.cnn_filters = cnn_filters
        self.fc_size = fc_size
        self.sequence_length = sequence_length
        self.dropout_p = dropout_p
        self.batch_size = batch_size
        self.optimizer_type = optimizer_type
        self.lr = lr        
        self.weight_decay = weight_decay
        self.sgd_momentum = sgd_momentum
        self.loss = loss
        self.bce_pos_class_weight = bce_pos_class_weight
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        self.num_epochs = num_epochs
        self.cv_folds = cv_folds
        self.scheduler_t = scheduler_t

    def reshape_data_to_overlapping_seq(self, data, seq_len):
        num_samples, input_features = data.shape
        num_sequences = num_samples - seq_len + 1
        sequences = np.zeros((num_sequences, seq_len, input_features), dtype=data.dtype)
        for i in range(num_sequences):
            sequences[i] = data[i:i + seq_len]
        return sequences
    
    def fit(self, X, y):

        cv_tss_values = []
        cv_hss_values = []

        tscv = TimeSeriesSplit(n_splits=self.cv_folds)
        for i, (train_index, val_index) in enumerate(tscv.split(X)):
            
            model = simple_timeSeries_CNN(cnn_filters=self.cnn_filters, 
                                          fc_size=self.fc_size, 
                                          dropout_p=self.dropout_p, 
                                          loss=self.loss, 
                                          sequence_length=self.sequence_length)
            model.to(device)
            
            if self.loss == 'bce':
                criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([self.bce_pos_class_weight]).to(device))
            elif self.loss == 'focal':
                criterion = FocalLoss(alpha=self.focal_alpha, gamma=self.focal_gamma)

            if self.optimizer_type == 'adam':
                optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            elif self.optimizer_type == 'sgd':
                optimizer = optim.SGD(model.parameters(), lr=self.lr, momentum=self.sgd_momentum, weight_decay=self.weight_decay)

            if self.scheduler_t > 0:
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.scheduler_t)
            else:
                scheduler = False

            X_train, X_val = X[train_index], X[val_index]
            y_train, y_val = y[train_index], y[val_index]
            
            scaler_X = RobustScaler()
            scaler_X = scaler_X.fit(X_train)

            X_train_scaled = scaler_X.transform(X_train)
            X_val_scaled = scaler_X.transform(X_val)
            
            X_train_scaled = self.reshape_data_to_overlapping_seq(X_train_scaled, self.sequence_length)
            X_val_scaled = self.reshape_data_to_overlapping_seq(X_val_scaled, self.sequence_length)
            
            train_inputs = torch.tensor(X_train_scaled, dtype=torch.float32).transpose(1, 2)
            train_labels = torch.tensor(y_train[self.sequence_length - 1:], dtype=torch.float32).unsqueeze(1)
            train_dataset = TensorDataset(train_inputs, train_labels)
            train_dataloader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)

            val_inputs = torch.tensor(X_val_scaled, dtype=torch.float32).transpose(1, 2)
            val_labels = torch.tensor(y_val[self.sequence_length - 1:], dtype=torch.float32).unsqueeze(1)
            val_dataset = TensorDataset(val_inputs, val_labels)
            val_dataloader = DataLoader(val_dataset, batch_size=self.batch_size, shuffle=False, worker_init_fn=seed_worker)
            
            training_hss, validation_hss, training_tss, validation_tss = \
            train_model_with_cv(model, 
                 train_dataloader, val_dataloader, 
                 criterion, optimizer, scheduler,
                 num_epochs=self.num_epochs)
            cv_tss_values.append(validation_tss[-1])
            cv_hss_values.append(validation_hss[-1])

            if i == 0:
                total_params = sum(p.numel() for p in model.parameters())
                print(f'Total number of parameters: {total_params}')
                
            print(f"Fold {i}: train tss = {training_tss[-1]:.4f} : val tss = {validation_tss[-1]:.4f} ::: train hss = {training_hss[-1]:.4f} : val hss = {validation_hss[-1]:.4f}")
        
        return cv_hss_values, cv_tss_values
    

In [10]:
training_data_size = 50000

X_train, X_test, \
    y_train, y_test, \
        idx_train, idx_test = train_test_split(X_fSelect, y, range(len(y)), train_size=training_data_size, shuffle=False)

# ### small test
# param_grid = {
#     'cnn_filters': [4],   
#     'fc_size': [64, 256],  
#     'sequence_length': [120],   
#     'dropout_p': [0.3],  
#     'batch_size': [512],  
#     'lr': [0.001],  
#     'optimizer_type': ['adam'],  
#     'weight_decay':[0.01],  
#     'loss': ['bce'],
#     'bce_pos_class_weight': [30],  
#     'num_epochs': [15],  
#     'cv_folds': [5], 
# }

### strategic hypertune iteration 1
param_grid = {
    'cnn_filters': [4, 32],
    'fc_size': [64, 1048],
    'sequence_length': [60, 120, 180],
    'dropout_p': [0.1, 0.3],
    'batch_size': [1024, 2048],
    'lr': [0.001, 0.01],
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.001, 0.01],
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 60],
#     'scheduler_t': [5],
    'num_epochs': [15],
    'cv_folds': [5], 
}

### strategic hypertune iteration 2
param_grid = {
    'cnn_filters': [4, 16, 64],  ### changed, added 1 dimension
    'fc_size': [128, 2048],  ### changed
    'sequence_length': [60, 180],  ### changed, removed 1 dimension
    'dropout_p': [0.1, 0.3],
    'batch_size': [2048, 4096],  ### changed
    'lr': [0.0075, 0.025],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.001, 0.01],
    'loss': ['bce'],
    'bce_pos_class_weight': [30, 60],  ### changed
#     'scheduler_t': [5],
    'num_epochs': [15],
    'cv_folds': [5], 
}

### strategic hypertune iteration 3
param_grid = {
    'cnn_filters': [64, 256],  ### changed, removed 1 dimension
    'fc_size': [512, 2048],  ### changed
    'sequence_length': [60, 120, 300],  ### changed, added 1 dimension
    'dropout_p': [0.1, 0.3],
    'batch_size': [1024, 4096],  ### changed
    'lr': [0.005, 0.05],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.001, 0.01],
    'loss': ['bce'],
    'bce_pos_class_weight': [20, 60],  ### changed
#     'scheduler_t': [5],
    'num_epochs': [15],
    'cv_folds': [5], 
}

### strategic hypertune iteration 4
param_grid = {
    'cnn_filters': [8, 32],  ### changed
    'fc_size': [128, 512],  ### changed
    'sequence_length': [30, 60, 120],  ### changed
    'dropout_p': [0.1, 0.3],
    'batch_size': [2048, 4096],  ### changed
    'lr': [0.0075, 0.03],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.001, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [15, 50],  ### changed
#     'scheduler_t': [5],
    'num_epochs': [15],
    'cv_folds': [5], 
}

### strategic hypertune iteration 5
param_grid = {
    'cnn_filters': [16, 48],  ### changed
    'fc_size': [256, 768],  ### changed
    'sequence_length': [30, 60, 120],  
    'dropout_p': [0.05, 0.4],  ### changed
    'batch_size': [3584, 4864],  ### changed
    'lr': [0.005, 0.025],  ### changed
    'optimizer_type': ['adam', 'sgd'],
    'weight_decay':[0.0025, 0.05],  ### changed
    'loss': ['bce'],
    'bce_pos_class_weight': [15, 50],  
#     'scheduler_t': [5],
    'num_epochs': [15],
    'cv_folds': [5], 
}

param_combinations = list(ParameterGrid(param_grid))

grid = pd.DataFrame()
tss_distributions = []
hss_distributions = []
mean_tss_scores = []
mean_hss_scores = []
for params in param_combinations:

    print(str(param_combinations.index(params)) + "/" + str(len(param_combinations)))
    print(params)

    cnn_time_series_classifier = CNNBinaryTimeSeriesClassifier(**params)
    cv_hss_values, cv_tss_values = cnn_time_series_classifier.fit(X_train, y_train)
    
    tss_distributions.append(cv_tss_values)
    hss_distributions.append(cv_hss_values)
    mean_cv_tss = sum(cv_tss_values)/len(cv_tss_values)
    mean_cv_hss = sum(cv_hss_values)/len(cv_hss_values)
    print(f"mean cv TSS: {mean_cv_tss:.4f}")
    print(f"mean cv HSS: {mean_cv_hss:.4f}")
    mean_tss_scores.append(mean_cv_tss)
    mean_hss_scores.append(mean_cv_hss)
    
    if params == param_combinations[0]:
        grid = pd.DataFrame(params, index=[0])
    else:
        grid = pd.concat([grid, pd.DataFrame([params])], ignore_index=True)

grid['tss'] = [round(num, 4) for num in mean_tss_scores]
grid['hss'] = [round(num, 4) for num in mean_hss_scores]
grid = grid.sort_values(by=['tss', 'hss'], ascending=False)
grid = grid.drop(columns=[col for col in ['num_epochs','cv_folds','loss','scheduler_t'] if col in grid.columns], axis=1)
grid.head(20)


0/768
{'batch_size': 3584, 'bce_pos_class_weight': 15, 'cnn_filters': 16, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 256, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 15, 'optimizer_type': 'adam', 'sequence_length': 30, 'weight_decay': 0.0025}
Total number of parameters: 60257
Fold 0: train tss = 0.0000 : val tss = -0.0001 ::: train hss = 0.0000 : val hss = -0.0002
Fold 1: train tss = 0.7628 : val tss = 0.0987 ::: train hss = 0.5894 : val hss = 0.0686
Fold 2: train tss = 0.6556 : val tss = 0.6117 ::: train hss = 0.3959 : val hss = 0.3510
Fold 3: train tss = 0.7061 : val tss = 0.0237 ::: train hss = 0.4946 : val hss = 0.0244
Fold 4: train tss = 0.7266 : val tss = 0.5673 ::: train hss = 0.3802 : val hss = 0.4174
mean cv TSS: 0.2603
mean cv HSS: 0.1722
1/768
{'batch_size': 3584, 'bce_pos_class_weight': 15, 'cnn_filters': 16, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 256, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 15, 'optimizer_type': 'adam', 'sequence_length': 30, 'weight_decay': 0.05}


Fold 4: train tss = 0.7029 : val tss = 0.2800 ::: train hss = 0.4706 : val hss = 0.2605
mean cv TSS: 0.2324
mean cv HSS: 0.1397
11/768
{'batch_size': 3584, 'bce_pos_class_weight': 15, 'cnn_filters': 16, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 256, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 15, 'optimizer_type': 'sgd', 'sequence_length': 120, 'weight_decay': 0.05}
Total number of parameters: 248673
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.7759 : val tss = 0.1724 ::: train hss = 0.6778 : val hss = 0.0640
Fold 2: train tss = 0.5086 : val tss = 0.6081 ::: train hss = 0.4202 : val hss = 0.2797
Fold 3: train tss = 0.7826 : val tss = 0.0251 ::: train hss = 0.4868 : val hss = 0.0428
Fold 4: train tss = 0.7527 : val tss = 0.4841 ::: train hss = 0.4572 : val hss = 0.4461
mean cv TSS: 0.2579
mean cv HSS: 0.1665
12/768
{'batch_size': 3584, 'bce_pos_class_weight': 15, 'cnn_filters': 16, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_si

Fold 3: train tss = 0.6715 : val tss = 0.0435 ::: train hss = 0.3315 : val hss = 0.0347
Fold 4: train tss = 0.6034 : val tss = 0.5666 ::: train hss = 0.3097 : val hss = 0.4161
mean cv TSS: 0.3105
mean cv HSS: 0.1802
22/768
{'batch_size': 3584, 'bce_pos_class_weight': 15, 'cnn_filters': 16, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 256, 'loss': 'bce', 'lr': 0.025, 'num_epochs': 15, 'optimizer_type': 'sgd', 'sequence_length': 120, 'weight_decay': 0.0025}
Total number of parameters: 248673
Fold 0: train tss = 0.2494 : val tss = 0.0000 ::: train hss = 0.3325 : val hss = 0.0000
Fold 1: train tss = 0.7964 : val tss = 0.1065 ::: train hss = 0.6447 : val hss = 0.0578
Fold 2: train tss = 0.6096 : val tss = 0.5659 ::: train hss = 0.4641 : val hss = 0.3556
Fold 3: train tss = 0.7603 : val tss = 0.0510 ::: train hss = 0.5326 : val hss = 0.0140
Fold 4: train tss = 0.7235 : val tss = 0.2410 ::: train hss = 0.3966 : val hss = 0.2327
mean cv TSS: 0.1929
mean cv HSS: 0.1320
23/768
{'batch_size': 358

Fold 2: train tss = 0.7044 : val tss = 0.6157 ::: train hss = 0.4507 : val hss = 0.4529
Fold 3: train tss = 0.7728 : val tss = 0.0264 ::: train hss = 0.4663 : val hss = 0.0141
Fold 4: train tss = 0.7383 : val tss = 0.4533 ::: train hss = 0.4070 : val hss = 0.3468
mean cv TSS: 0.2763
mean cv HSS: 0.1885
33/768
{'batch_size': 3584, 'bce_pos_class_weight': 15, 'cnn_filters': 16, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 768, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 15, 'optimizer_type': 'sgd', 'sequence_length': 60, 'weight_decay': 0.05}
Total number of parameters: 372577
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.7708 : val tss = 0.3063 ::: train hss = 0.6616 : val hss = 0.1294
Fold 2: train tss = 0.7105 : val tss = 0.5592 ::: train hss = 0.4194 : val hss = 0.4592
Fold 3: train tss = 0.6821 : val tss = 0.0879 ::: train hss = 0.4133 : val hss = 0.0203
Fold 4: train tss = 0.6371 : val tss = 0.4933 ::: train hss = 0.2877

Fold 1: train tss = 0.7572 : val tss = 0.1845 ::: train hss = 0.5692 : val hss = 0.1864
Fold 2: train tss = 0.5328 : val tss = 0.5655 ::: train hss = 0.2909 : val hss = 0.3984
Fold 3: train tss = 0.5988 : val tss = 0.0652 ::: train hss = 0.3181 : val hss = 0.0232
Fold 4: train tss = 0.5391 : val tss = 0.7204 ::: train hss = 0.2424 : val hss = 0.3015
mean cv TSS: 0.3071
mean cv HSS: 0.1819
44/768
{'batch_size': 3584, 'bce_pos_class_weight': 15, 'cnn_filters': 16, 'cv_folds': 5, 'dropout_p': 0.05, 'fc_size': 768, 'loss': 'bce', 'lr': 0.025, 'num_epochs': 15, 'optimizer_type': 'sgd', 'sequence_length': 60, 'weight_decay': 0.0025}
Total number of parameters: 372577
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.7673 : val tss = 0.1485 ::: train hss = 0.6810 : val hss = 0.1008
Fold 2: train tss = 0.7155 : val tss = 0.7183 ::: train hss = 0.4068 : val hss = 0.4144
Fold 3: train tss = 0.6151 : val tss = 0.1262 ::: train hss = 0.43

...

Total number of parameters: 754209
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.8262 : val tss = 0.0585 ::: train hss = 0.3296 : val hss = 0.0219
Fold 2: train tss = 0.6602 : val tss = 0.6041 ::: train hss = 0.1190 : val hss = 0.2157
Fold 3: train tss = 0.6060 : val tss = 0.3275 ::: train hss = 0.1441 : val hss = 0.0343
Fold 4: train tss = 0.6016 : val tss = 0.6708 ::: train hss = 0.1050 : val hss = 0.2322
mean cv TSS: 0.3322
mean cv HSS: 0.1008
726/768
{'batch_size': 4864, 'bce_pos_class_weight': 50, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.4, 'fc_size': 256, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 15, 'optimizer_type': 'sgd', 'sequence_length': 30, 'weight_decay': 0.0025}
Total number of parameters: 188961
Fold 0: train tss = 0.0000 : val tss = -0.0181 ::: train hss = 0.0000 : val hss = -0.0210
Fold 1: train tss = 0.7847 : val tss = 0.2982 ::: train hss = 0.2912 : val hss = 0.0813
Fold 2: train tss = 0.6881 : 

Total number of parameters: 754209
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.7049 : val tss = 0.0881 ::: train hss = 0.2560 : val hss = 0.0283
Fold 2: train tss = 0.5107 : val tss = 0.5717 ::: train hss = 0.0589 : val hss = 0.1977
Fold 3: train tss = 0.5729 : val tss = 0.4886 ::: train hss = 0.0991 : val hss = 0.0376
Fold 4: train tss = 0.5894 : val tss = 0.5706 ::: train hss = 0.1120 : val hss = 0.1971
mean cv TSS: 0.3438
mean cv HSS: 0.0921
737/768
{'batch_size': 4864, 'bce_pos_class_weight': 50, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.4, 'fc_size': 256, 'loss': 'bce', 'lr': 0.025, 'num_epochs': 15, 'optimizer_type': 'adam', 'sequence_length': 120, 'weight_decay': 0.05}
Total number of parameters: 754209
Fold 0: train tss = 0.1680 : val tss = -0.0059 ::: train hss = 0.0235 : val hss = -0.0095
Fold 1: train tss = 0.6612 : val tss = 0.1151 ::: train hss = 0.0529 : val hss = 0.0147
Fold 2: train tss = 0.5121 : 

Total number of parameters: 1123873
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.7517 : val tss = 0.2671 ::: train hss = 0.2757 : val hss = 0.0987
Fold 2: train tss = 0.5517 : val tss = 0.6323 ::: train hss = 0.0843 : val hss = 0.2410
Fold 3: train tss = 0.7568 : val tss = 0.2442 ::: train hss = 0.2308 : val hss = 0.0388
Fold 4: train tss = 0.6004 : val tss = 0.4562 ::: train hss = 0.1206 : val hss = 0.0986
mean cv TSS: 0.3200
mean cv HSS: 0.0954
748/768
{'batch_size': 4864, 'bce_pos_class_weight': 50, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.4, 'fc_size': 768, 'loss': 'bce', 'lr': 0.005, 'num_epochs': 15, 'optimizer_type': 'adam', 'sequence_length': 120, 'weight_decay': 0.0025}
Total number of parameters: 2229793
Fold 0: train tss = 0.2480 : val tss = -0.0027 ::: train hss = 0.2420 : val hss = -0.0049
Fold 1: train tss = 0.8547 : val tss = 0.1716 ::: train hss = 0.2575 : val hss = 0.0767
Fold 2: train tss = 0.682

Total number of parameters: 1123873
Fold 0: train tss = 0.4313 : val tss = -0.0337 ::: train hss = 0.0943 : val hss = -0.0286
Fold 1: train tss = 0.6926 : val tss = 0.1408 ::: train hss = 0.0868 : val hss = 0.0275
Fold 2: train tss = 0.4413 : val tss = 0.5492 ::: train hss = 0.0343 : val hss = 0.1484
Fold 3: train tss = 0.6539 : val tss = 0.1063 ::: train hss = 0.1288 : val hss = 0.0213
Fold 4: train tss = 0.6069 : val tss = 0.6361 ::: train hss = 0.1115 : val hss = 0.2012
mean cv TSS: 0.2798
mean cv HSS: 0.0740
759/768
{'batch_size': 4864, 'bce_pos_class_weight': 50, 'cnn_filters': 48, 'cv_folds': 5, 'dropout_p': 0.4, 'fc_size': 768, 'loss': 'bce', 'lr': 0.025, 'num_epochs': 15, 'optimizer_type': 'adam', 'sequence_length': 60, 'weight_decay': 0.05}
Total number of parameters: 1123873
Fold 0: train tss = 0.0000 : val tss = 0.0000 ::: train hss = 0.0000 : val hss = 0.0000
Fold 1: train tss = 0.5943 : val tss = 0.1996 ::: train hss = 0.0570 : val hss = 0.0329
Fold 2: train tss = 0.5131 :

,batch_size,bce_pos_class_weight,cnn_filters,dropout_p,fc_size,lr,optimizer_type,sequence_length,weight_decay,tss,hss
336,3584,50,48,0.40,256,0.005,adam,30,0.0025,0.4354,0.1377
753,4864,50,48,0.40,768,0.005,sgd,60,0.0500,0.4242,0.1585
675,4864,50,48,0.05,256,0.005,adam,60,0.0500,0.4167,0.1705
340,3584,50,48,0.40,256,0.005,adam,120,0.0025,0.4142,0.1262
650,4864,50,16,0.40,768,0.005,adam,60,0.0025,0.4124,0.1190
271,3584,50,16,0.40,768,0.005,sgd,30,0.0500,0.4109,0.1191
722,4864,50,48,0.40,256,0.005,adam,60,0.0025,0.4099,0.1453
667,4864,50,16,0.40,768,0.025,sgd,30,0.0500,0.4020,0.1424
362,3584,50,48,0.40,768,0.005,adam,60,0.0025,0.4010,0.1234
613,4864,50,16,0.05,768,0.025,adam,30,0.0500,0.4001,0.0962


In [11]:
grid.sort_values(by=['hss', 'tss'], ascending=False).head(20)

,batch_size,bce_pos_class_weight,cnn_filters,dropout_p,fc_size,lr,optimizer_type,sequence_length,weight_decay,tss,hss
97,3584,15,48,0.05,256,0.005,adam,30,0.0500,0.3197,0.2223
34,3584,15,16,0.05,768,0.005,sgd,120,0.0025,0.3695,0.2187
402,4864,15,16,0.05,256,0.025,sgd,30,0.0025,0.3221,0.2176
526,4864,15,48,0.05,768,0.025,sgd,120,0.0025,0.2634,0.2172
44,3584,15,16,0.05,768,0.025,sgd,60,0.0025,0.3461,0.2139
549,4864,15,48,0.40,256,0.025,sgd,60,0.0500,0.3205,0.2139
35,3584,15,16,0.05,768,0.005,sgd,120,0.0500,0.2874,0.2136
416,4864,15,16,0.05,768,0.005,sgd,60,0.0025,0.2909,0.2113
482,4864,15,48,0.05,256,0.005,adam,60,0.0025,0.3144,0.2108
394,4864,15,16,0.05,256,0.005,sgd,120,0.0025,0.3100,0.2103


In [12]:
if grid.empty:
    print("grid empty")
else:
    grid.to_csv("cnn_overlap_hypertune_grid_output_5.csv")
    print("grid output successful, saved as cnn_overlap_hypertune_grid_output_5.csv")
    

grid output successful, saved as cnn_overlap_hypertune_grid_output_5.csv
